# Thesis scoring on Kaggle
Judge pass of score.py over all 21,600 runs. Input: private dataset `thesis-score-bundle`. GPU T4 x2, Internet ON. Output: `/kaggle/working/results/scores_*.jsonl`.

In [ ]:
GROUPS = {   # out-file suffix -> extra score.py args. Judges from config.yaml: qwen2.5:7b default, mistral:7b for qwen.
    'g1': ['--runs', 'runs/llama3_*.jsonl', 'runs/mistral_*.jsonl'],
    'g2': ['--runs', 'runs/gemma_*.jsonl', 'runs/biomistral_*.jsonl'],
    'g3': ['--runs', 'runs/qwen_*.jsonl', 'runs/meditron_*.jsonl'],
    # judge calibration: a second judge on the same rows, compared with qwen2.5:7b in analyze.py (agreement / kappa)
    'calib_mistral': ['--runs', 'runs/llama3_baseline.jsonl', 'runs/gemma_baseline.jsonl', '--judge', 'mistral:7b'],
}

In [ ]:
import glob, os, shutil, zipfile
zips = glob.glob('/kaggle/input/*/thesis_score.zip')
if zips:
    zipfile.ZipFile(zips[0]).extractall('/kaggle/working')
else:
    src = os.path.dirname(glob.glob('/kaggle/input/**/score.py', recursive=True)[0])
    shutil.copytree(src, '/kaggle/working/thesis', dirs_exist_ok=True)
os.chdir('/kaggle/working/thesis')
os.makedirs('results', exist_ok=True); os.makedirs('/kaggle/working/results', exist_ok=True)
for f in glob.glob('/kaggle/working/results/scores_*.jsonl'):      # resume from a previous version
    dst = os.path.join('results', os.path.basename(f))
    if not os.path.exists(dst): shutil.copy(f, dst)
print(sorted(os.listdir('.')), len(glob.glob('runs/*.jsonl')), 'run files')

In [ ]:
%%bash
pip -q install ollama pyyaml zstandard
mkdir -p /kaggle/working/ollama
curl -sL https://github.com/ollama/ollama/releases/download/v0.40.16/ollama-linux-amd64.tar.zst -o /tmp/ollama.tar.zst
python -c "import zstandard, tarfile; tarfile.open(fileobj=zstandard.ZstdDecompressor().stream_reader(open('/tmp/ollama.tar.zst','rb')), mode='r|').extractall('/kaggle/working/ollama')"
/kaggle/working/ollama/bin/ollama --version
nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess, time, yaml, urllib.request
os.environ['PATH'] = '/kaggle/working/ollama/bin:' + os.environ['PATH']
os.environ['OLLAMA_FLASH_ATTENTION'] = '1'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '2'      # the two judge models, one per GPU
os.environ['OLLAMA_NUM_PARALLEL'] = '4'           # batch the concurrent judge requests
server = subprocess.Popen(['ollama', 'serve'], stdout=open('/kaggle/working/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2); break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit(open('/kaggle/working/ollama.log').read()[-2000:])
cfg = yaml.safe_load(open('config.yaml'))
for tag in sorted(set(cfg['scoring']['judge'].values())):
    print('pulling', tag, flush=True)
    subprocess.run(['ollama', 'pull', tag], check=True)
print(subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout)

In [ ]:
import subprocess, sys, shutil, os, time, glob
procs = {g: subprocess.Popen([sys.executable, '-u', 'score.py', '--out', f'results/scores_{g}.jsonl', *args],
                             stdout=open(f'/kaggle/working/score_{g}.log', 'w'), stderr=subprocess.STDOUT)
         for g, args in GROUPS.items()}
t0 = time.time()
def sync():
    for f in glob.glob('results/scores_*.jsonl'):
        shutil.copy(f, '/kaggle/working/results/' + os.path.basename(f))
while any(p.poll() is None for p in procs.values()):
    time.sleep(180); sync()
    done = {g: sum(1 for _ in open(f'results/scores_{g}.jsonl')) if os.path.exists(f'results/scores_{g}.jsonl') else 0 for g in procs}
    print(f'{(time.time()-t0)/60:5.1f} min', done, flush=True)
sync()
print('exit codes', {g: p.returncode for g, p in procs.items()})
for g in procs:
    print(g, open(f'/kaggle/working/score_{g}.log').read()[-500:])

In [ ]:
!wc -l /kaggle/working/results/*.jsonl